# 07 · How does a model *write* about an image?

So far CLIP could only *pick* things, never write them. This time we build a tiny captioner from scratch that looks at a photo and writes a caption one token at a time, using **cross-attention** to read the image as it goes.

**Read more in [the book](https://arxiv.org/abs/2605.07544):** Chapter 3, Section 3.2, *Conditioning Language Models on Visual Input*, and Chapter 4, Section 4.1.1, *Encoder-Decoder Captioning*.

> **How to use this notebook:** Run the cells top to bottom. When you hit a **Pause and predict**, take a guess before running the next cell. You'll remember way more that way. Cells marked **TODO** are yours to play with: change a value, re-run, and see what happens.

### Pick your GPU

Got a GPU (or a few)? Set `GPU_ID` in the next cell to the one you want. Run `nvidia-smi` in a terminal to see the list. Only have one? Leave it at `"0"`. No GPU at all? Set it to `""` and everything runs on the CPU. It's slower, but it works.

This cell has to run *before* PyTorch is imported, so if you change it later, restart the kernel and run from the top.

In [ ]:
# TODO: choose your GPU. This has to run before torch is imported.
#   "0"  -> the first GPU (leave this if you only have one)
#   "1"  -> the second GPU, and so on (`nvidia-smi` lists them)
#   ""   -> no GPU, run everything on the CPU
#   None -> don't restrict anything; PyTorch sees every GPU
GPU_ID = "0"

import os
if GPU_ID is not None:
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # number GPUs the same way nvidia-smi does
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 1. What are we teaching the model to describe?

Before any model shows up, we need data: photos with captions that people wrote for them.

### 1.1 Grab our tools

The imports cover four jobs: loading a dataset, prepping images, tokenizing captions, and building the decoder.

In [ ]:
from textwrap import fill  # wraps long captions so they fit above each plot
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from datasets import load_dataset
from torch import nn
from torchvision.transforms import CenterCrop, Compose, Resize, ToTensor
from transformers import AutoTokenizer  # a real pretrained tokenizer, no hand-rolled word splitting

### 1.2 Keep everything tiny

We'll use sixteen pairs and 32 × 32 images. That's never going to give us a good captioner, and that's not the point. At this size we can actually look at every shape and every number the model touches.

First, which dataset? A small slice of COCO Captions, pinned to an exact revision so the rows can't change under us later.

In [ ]:
DATASET_ID = "Erland/coco_captions_small"  # pinned revision = the exact same rows every run
DATASET_REVISION = "a516b5101cec691eb65f455ffdb054afb2fdd5dd"

Now the knobs. These few values decide most of the shapes in this notebook:

- `PAIR_COUNT`: how many image–caption pairs we train on (16)
- `IMAGE_SIZE` and `PATCH_SIZE`: 32 × 32 images, cut into 8 × 8 patches. How many patches is that per side? Keep a guess in mind; we'll check it in Section 3.
- `MAX_LENGTH`: how many token positions each caption gets (18)
- `MODEL_WIDTH`: how many numbers describe each image patch *and* each text token inside the decoder (48)
- `EPOCHS`: how many training updates we'll do (300)

In [ ]:
PAIR_COUNT, IMAGE_SIZE, PATCH_SIZE = 16, 32, 8  # small on purpose, so every sequence is easy to inspect
MAX_LENGTH, MODEL_WIDTH, EPOCHS = 18, 48, 300

A fixed **seed** makes the random starting weights the same every run. The **learning rate** sets how big a step the optimizer takes each time it updates those weights.

In [ ]:
SEED, LEARNING_RATE = 7, 0.005  # same seed, same starting weights every run
torch.manual_seed(SEED)

Everything lives on one device: the GPU if we have one, the CPU otherwise. Printing it now catches a setup mistake before we load anything big.

> **Heads up:** Expected a GPU but this says `cpu`? Check `GPU_ID` in the first code cell. The model is tiny, so the CPU works too, just slower.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU if we have one, CPU otherwise
print("device:", device)

### 1.3 Load captions written for real images

Time to load the data. COCO has several human-written captions for most photos, so the same photo shows up in more than one row. We'll deal with that in a sec.

In [ ]:
dataset = load_dataset(  # pinned revision, so the rows never drift
    DATASET_ID, split="train", revision=DATASET_REVISION
)
print(dataset)

### 1.4 Keep one caption per photo

Each row has a `cocoid`, which identifies the photo. Rows with the same `cocoid` are the same picture with a different caption. To keep things simple, we want exactly one caption per photo.

Two dictionaries will do it: one remembers the first row we see for each photo, the other remembers its first caption.

In [ ]:
first_row = {}  # cocoid -> first row index for that photo
first_caption = {}  # cocoid -> first caption for that photo

`setdefault` only writes a value the first time it sees a key. So when a photo comes back with another caption, the first one stays put. The extra captions are still in the dataset; we just don't use them.

In [ ]:
for index, (cocoid, caption) in enumerate(zip(dataset["cocoid"], dataset["caption"])):
    first_row.setdefault(cocoid, index)  # repeat IDs are the same photo with another caption
    first_caption.setdefault(cocoid, caption.strip())

Our batch is the first sixteen distinct photos. Python dictionaries keep the order things were added in, so this picks the same sixteen every time. Nothing random here.

In [ ]:
sample_ids = list(first_row)[:PAIR_COUNT]  # 16 different photos, no repeats

### 1.5 Make every image the same shape

To stack images into one batch, they all need the same height and width. The transform does three things: resize so the short side is 40 pixels, crop the middle 32 × 32, and turn the result into a tensor in channels-first order, `[channels, height, width]`.

In [ ]:
image_transform = Compose([Resize(40), CenterCrop(IMAGE_SIZE), ToTensor()])  # same size for everyone -> same patch grid
records = [dataset[first_row[cocoid]] for cocoid in sample_ids]

Both the images and the captions are built from the same `sample_ids` list, in the same order. That matters a lot: row 0 of the images has to go with caption 0. Mix up the order and the model learns to describe the wrong pictures, and nothing will warn you.

In [ ]:
images = torch.stack([image_transform(record["image"].convert("RGB")) for record in records])  # both follow sample_ids,
captions = [first_caption[cocoid] for cocoid in sample_ids]  # so images[i] still goes with captions[i]

Read the shape left to right. You should see 16 images, 3 color channels, and 32 × 32 pixels. If anything's different, the patch math later on will be different too.

In [ ]:
print("image batch:", tuple(images.shape))  # [batch, channels, height, width]

Before building any model, let's eyeball four pairs. Two questions: does each caption actually match its photo, and is it short enough to fit in 18 tokens? This is a data check, not a model result yet.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3.5))  # each photo with its caption as the title
for axis, image, caption in zip(axes, images[:4], captions[:4]):
    axis.imshow(image.permute(1, 2, 0))
    axis.set_title(fill(caption, 25), fontsize=8)
    axis.axis("off")
fig.suptitle("Four real image–caption pairs")
fig.tight_layout()
plt.show()

Yep, 32 × 32 is *tiny*. You can usually still make out the scene, but fine detail is gone. Keep that in mind when you read the captions our model writes later.

## 2. How does a caption become something a model can predict?

A decoder doesn't write whole sentences in one go. It predicts **token IDs**, one at a time, where each ID is a slot in a fixed vocabulary. So first we need to turn captions into IDs.

### 2.1 Borrow a real tokenizer

We'll reuse CLIP's tokenizer, the same byte-pair encoding (BPE) one from Notebook 02. It already knows how to split words, punctuation, and rare words into pieces, so we don't have to invent our own rules.

> **Heads up:** Only the *tokenizer* comes from CLIP. The captioning model itself starts from random weights, and its image side works on raw pixel patches, not CLIP image features.

One more thing. CLIP normally uses its end-of-text token for padding too. We add our own separate **pad token**, and you'll see why in Section 2.3.

In [ ]:
TOKENIZER_ID = "openai/clip-vit-base-patch32"  # CLIP's BPE tokenizer
TOKENIZER_REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"  # pinned, so token IDs never change
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_ID,
    revision=TOKENIZER_REVISION,
)
tokenizer.add_special_tokens({"pad_token": "<|pad|>"})  # a brand-new token just for padding

### 2.2 Look at the pieces before the IDs

Let's tokenize the first caption from the gallery. Common words usually stay in one piece, while rarer words can get split up. That `</w>` stuck on the end of pieces is CLIP's way of marking "a word ends here". Nobody typed it; it's part of the vocabulary.

In [ ]:
sample_tokens = tokenizer.tokenize(captions[0])  # split using CLIP's learned merge rules
print("caption:", captions[0])
print("tokens:", sample_tokens)

**TODO:** Type your own caption and see how it gets chopped up. Try a rare word, a made-up word, or some punctuation.

In [ ]:
# TODO: write any caption you like
my_caption = "a tabby cat napping on a bright red sofa."
print("tokens:", tokenizer.tokenize(my_caption))

Made-up words usually fall apart into lots of tiny pieces. That's the nice thing about BPE: there's no such thing as an "unknown word", just more pieces.

### 2.3 Meet the control tokens

Three IDs don't stand for words; they control the sequence:

- **BOS** (beginning of sequence) opens every caption.
- **EOS** (end of sequence) says "the caption is done".
- **PAD** fills empty slots so every caption has the same length.

Giving them names now makes the training and generation code way easier to read.

In [ ]:
BOS_ID = tokenizer.bos_token_id  # BOS starts a caption, EOS ends it
EOS_ID = tokenizer.eos_token_id
PAD_ID = tokenizer.pad_token_id  # our new pad token, separate from EOS

So why a separate pad token? Because ending and filling need different treatment. EOS is something the model must learn to predict, otherwise it never knows when to stop. PAD positions are just filler, and they shouldn't count toward the loss at all. If both shared one ID, ignoring PAD in the loss would also throw away every EOS.

Let's print the IDs. Our pad token was added on top of CLIP's existing vocabulary, so it gets a brand-new ID at the very end, and the vocabulary grows by one.

In [ ]:
print("tokenizer class:", type(tokenizer).__name__)  # these come from the pinned checkpoint files
print("vocabulary size:", len(tokenizer))
print("BOS / EOS / PAD:", BOS_ID, EOS_ID, PAD_ID)

### 2.4 Pad everything to one length

Now we tokenize all sixteen captions at once. The tokenizer adds BOS and EOS, cuts off any caption that's too long, and pads the short ones up to `MAX_LENGTH`. The result is one neat rectangle of IDs, which keeps the next steps simple.

In [ ]:
# every row gets exactly MAX_LENGTH positions, so the whole batch fits in one tensor
caption_batch = tokenizer(
    captions,
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

We get two tensors back, lined up position by position:

- `input_ids`: the token IDs themselves
- `attention_mask`: `True` for real caption positions, `False` for padding

We keep the mask around, but we won't actually need it later. Our dedicated PAD ID is enough to spot padding in the loss.

In [ ]:
caption_ids = caption_batch["input_ids"]  # rows follow the same order as images
caption_mask = caption_batch["attention_mask"].bool()  # True = real caption token, False = padding

Turning the first row of IDs back into token strings is a good sanity check. Look for BOS first, then the caption pieces, then EOS, then a run of PAD tokens. The pieces in the middle should match what you saw in Section 2.2.

In [ ]:
first_tokens = tokenizer.convert_ids_to_tokens(caption_ids[0])  # IDs back to pieces, exactly as stored
print("caption batch:", tuple(caption_ids.shape))
print("first token row:", first_tokens)

### 2.5 Shift by one: what's the input, what's the target?

Here's the core trick behind training a caption decoder. Say a caption is `[BOS, a, dog, EOS]`. We feed the model `[BOS, a, dog]` and ask it to predict `[a, dog, EOS]`. Every position asks the same question: *given everything so far, what comes next?*

Notice that during training we always feed the model the **correct** previous tokens from the real caption, never its own guesses. That's called **teacher forcing**. It lets us train every position of every caption in a single forward pass.

In [ ]:
decoder_input = caption_ids[:, :-1]  # everything except the last position
next_token_target = caption_ids[:, 1:]  # everything except the first: the "next token" at each step

The two shapes should match, because every input position has exactly one target. Both are one shorter than `MAX_LENGTH`: 18 - 1 = 17.

The second number counts the target positions that actually get trained. It skips PAD but keeps EOS, so the model learns *when to stop*.

In [ ]:
print("input / target:", tuple(decoder_input.shape), tuple(next_token_target.shape))  # lined up position by position
print("supervised target positions:", next_token_target.ne(PAD_ID).sum().item())  # PAD doesn't count, EOS does

**TODO:** Let's see the shift for real. This prints each input token next to the token it's supposed to predict. Change `my_row` (0 to 15) to look at another caption.

In [ ]:
# TODO: pick any caption row from 0 to 15
my_row = 0
my_inputs = tokenizer.convert_ids_to_tokens(decoder_input[my_row].tolist())
my_targets = tokenizer.convert_ids_to_tokens(next_token_target[my_row].tolist())
for step, (seen, target) in enumerate(zip(my_inputs, my_targets)):
    print(f"step {step:2d}: sees {seen:>18} -> should predict {target}")

Each target shows up as the input one row below it. And once you hit EOS, the rest are just PAD predicting PAD, which the loss ignores.

## 3. How do the words get to look at the image?

We've got text ready. Now the image side. The decoder will need the image as a **sequence**, just like text, so it has something to attend to.

That wasn't always how captioning worked. One of the first neural captioners, *Show and Tell* ([Vinyals et al., 2015](https://arxiv.org/abs/1411.4555)), squeezed the whole photo into **one** CNN vector and handed it to an RNN once, at the start:

<img src="../assets/diagrams/notebook7/showtell_fig1.png" alt="Show and Tell: a vision CNN followed by a language-generating RNN" width="500">

*Figure 1 from Vinyals et al. (2015).*

The catch is that one vector has to hold everything the decoder will ever want to say. If we give the decoder a *sequence* of patches instead, it can look at a different part of the image for each word. That's what cross-attention is for.

### 3.1 Count the visual tokens

A 32 × 32 color image has 3 × 32 × 32 = 3,072 numbers. Instead of treating that as one long vector, we do what the Vision Transformer did in Notebook 01: cut the image into square, non-overlapping patches, and treat each patch as one token.

In [ ]:
PATCHES_PER_SIDE = IMAGE_SIZE // PATCH_SIZE  # patches along one side of the image
PATCH_COUNT, PATCH_WIDTH = PATCHES_PER_SIDE ** 2, 3 * PATCH_SIZE ** 2  # how many patches, numbers per patch

Two numbers to keep straight:

- `PATCH_COUNT`: how many visual tokens we get (the length of the image sequence)
- `PATCH_WIDTH`: how many raw numbers are in one patch (3 color channels × 8 × 8 pixels)

> **Pause and predict:** 32 × 32 images, 8 × 8 patches. How many visual tokens, and how many numbers in each one?

In [ ]:
print("visual tokens / patch values:", PATCH_COUNT, PATCH_WIDTH)  # each patch = RGB values from one square

32 ÷ 8 = 4 patches per side, so 4 × 4 = **16** visual tokens. Each one holds 3 × 8 × 8 = **192** raw pixel values. Soon a linear layer will squeeze those 192 numbers down to `MODEL_WIDTH` = 48.

### 3.2 Build the decoder

Our decoder is the right half of the original Transformer from [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762) (Vaswani et al., 2017). PyTorch's `nn.TransformerDecoderLayer` builds exactly that block.

<img src="../assets/diagrams/notebook7/transformer_fig1.png" alt="The Transformer model architecture" width="420">

*Figure 1 from Vaswani et al. (2017).*

Here's how the right half lines up with our captioner, from the bottom up:

- **Outputs (shifted right)** is the caption shifted by one token, our `decoder_input` from Section 2.5.
- **Masked Multi-Head Attention** is masked self-attention: each word only looks at the words before it.
- The middle **Multi-Head Attention** gets an arrow from the left half. That's **cross-attention**. In the paper, the left half reads a sentence in another language, because the model was built for translation. We don't have a left half at all. Our image patches plug straight into that arrow.
- **Linear** and **Softmax** at the top are our `head`, which scores every token in the vocabulary.
- **N×** means the block gets stacked. We use a single layer.

Each piece has one job:

- `unfold` cuts the image into patches.
- `patch` (for images) and `token` (for text) map both into the same width, 48. That's what lets text and image talk to each other.
- `image_position` and `text_position` are learned **position embeddings**: vectors that tell the model *where* each patch or token sits. Attention on its own has no sense of order.
- `decoder` is a standard Transformer decoder layer, and it does two kinds of attention:
  - **masked self-attention**: each text position looks at itself and the text tokens before it, never after.
  - **cross-attention**: each text position looks at *all 16* image patches. The text supplies the *queries* ("what am I looking for?"), and the image patches supply the *keys and values* ("here's what's in the picture"). This is the bridge from pixels to words.
- `head` turns each hidden state into one score per vocabulary token. Those scores are called **logits**.

> **Pause and predict:** Which of these pieces has the most parameters? Think about which ones have a size tied to the vocabulary.

In [ ]:
decoder_layer = nn.TransformerDecoderLayer(  # masked self-attention + cross-attention + a small MLP
    MODEL_WIDTH, nhead=4, dim_feedforward=96, dropout=0.0, batch_first=True
)
captioner = nn.ModuleDict({  # named pieces so we can poke at each one
    "unfold": nn.Unfold(PATCH_SIZE, stride=PATCH_SIZE),
    "patch": nn.Linear(PATCH_WIDTH, MODEL_WIDTH),
    "image_position": nn.Embedding(PATCH_COUNT, MODEL_WIDTH),
    "token": nn.Embedding(len(tokenizer), MODEL_WIDTH, padding_idx=PAD_ID),
    "text_position": nn.Embedding(MAX_LENGTH, MODEL_WIDTH),
    "decoder": nn.TransformerDecoder(decoder_layer, num_layers=1),
    "head": nn.Linear(MODEL_WIDTH, len(tokenizer)),
}).to(device)

Let's check your guess. This counts the parameters in each piece.

In [ ]:
for name, module in captioner.items():  # parameter count per piece
    print(f"{name:>15}: {sum(p.numel() for p in module.parameters()):>10,}")
print(f"{'total':>15}: {sum(p.numel() for p in captioner.parameters()):>10,}")

The two vocabulary-sized pieces, `token` and `head`, are far bigger than everything else. Each one stores 48 numbers for every token in the vocabulary (`head` adds one bias per token on top, so it edges ahead). The actual attention machinery is tiny next to them. `unfold` has zero parameters, since it just slices the image up.

### 3.3 Walk through one forward pass

Read the function in two halves:

1. **Image side:** pixels -> patches -> 48-wide vectors + positions. The result is called `memory`, the sequence the text gets to read from.
2. **Text side:** token IDs -> 48-wide vectors + positions.

Then the decoder mixes them. The **causal mask** is a grid of `True`/`False` values where `True` means "you can't look here". It hides future tokens, so position 3 can't peek at the answer at position 4. There's no mask on the image side, though: every text position can read every image patch.

In [ ]:
def caption_forward(image_batch, token_batch):
    patches = captioner["unfold"](image_batch).transpose(1, 2)  # [B, 3, 32, 32] -> [B, 16, 192]
    image_positions = torch.arange(PATCH_COUNT, device=image_batch.device)
    memory = captioner["patch"](patches) + captioner["image_position"](image_positions)  # [B, 16, 48], what every text step can read
    text_positions = torch.arange(token_batch.shape[1], device=token_batch.device)
    text = captioner["token"](token_batch) + captioner["text_position"](text_positions)  # [B, T, 48]
    steps = token_batch.shape[1]
    # True above the diagonal = "can't see future tokens"
    causal_mask = torch.triu(torch.ones(steps, steps, dtype=torch.bool, device=device), diagonal=1)
    hidden = captioner["decoder"](text, memory, tgt_mask=causal_mask)  # self-attention on text, cross-attention to memory
    return captioner["head"](hidden)  # [B, T, vocabulary]

### 3.4 Check the output shape

Let's push just two pairs through the untrained model. We don't care about the values yet; we only want to make sure the shape is right: one row per image, one position per decoder input, one score per vocabulary token.

First, though, we move images, inputs, and targets to our device together, so nothing ends up on the wrong device.

In [ ]:
images, decoder_input, next_token_target = (  # move all three together so nothing's left on the CPU
    images.to(device), decoder_input.to(device), next_token_target.to(device)
)
with torch.inference_mode():  # no gradients needed, just a shape check
    sample_logits = caption_forward(images[:2], decoder_input[:2])

> **Pause and predict:** The output is `[batch, time, vocabulary]`. We sent 2 images and decoder inputs that are 17 positions long. What are the first two numbers? And the last one?

In [ ]:
print("logits [batch, time, tokenizer vocabulary]:", tuple(sample_logits.shape))  # last axis = one score per token in the vocabulary

`[2, 17, ...]`, and the last number matches the vocabulary size we printed in Section 2.3. So the model gives a full set of vocabulary scores at every single position. **Cross-entropy**, the loss we'll use, compares each of those score rows with the one correct next-token ID, skipping PAD targets.

## 4. Can it learn our sixteen pairs?

### 4.1 Get a baseline loss

Before training, let's measure how bad the random model is. The exact number doesn't matter much. What matters is having something to compare against after training.

In [ ]:
initial_logits = caption_forward(images, decoder_input)  # all 16 pairs, untrained weights
initial_loss = F.cross_entropy(
    initial_logits.flatten(0, 1), next_token_target.flatten(), ignore_index=PAD_ID  # PAD targets don't count
)

Jot this number down. If training works, the loss should end up way below it. If it stays flat or goes up, the usual suspects are the one-position shift, the PAD handling in the loss, or the learning rate.

In [ ]:
print("initial loss:", round(initial_loss.item(), 4))  # our "before" number

### 4.2 The training loop

Every epoch does the same four things on the same sixteen pairs:

1. compute the vocabulary scores
2. measure the next-token cross-entropy
3. backpropagate the loss
4. update the captioner's weights

> **Heads up:** This is a deliberate **overfitting test**. We're asking "can this code learn sixteen examples by heart?", not "can it caption new photos?". If it can't even memorize sixteen pairs, something's broken.

In [ ]:
optimizer = torch.optim.Adam(captioner.parameters(), lr=LEARNING_RATE)  # updates every captioner weight
loss_history = []

We set up the optimizer and an empty loss list outside the loop, so it's easy to see what gets updated and what gets recorded. The loop itself runs all 16 pairs at once, 300 times. The model is tiny, so on a GPU this should be quick.

In [ ]:
captioner.train()  # same 16 pairs every epoch
for _ in range(EPOCHS):
    optimizer.zero_grad()
    logits = caption_forward(images, decoder_input)
    loss = F.cross_entropy(
        logits.flatten(0, 1), next_token_target.flatten(), ignore_index=PAD_ID
    )
    loss.backward()
    optimizer.step()
    loss_history.append(loss.detach().item())

> **Pause and predict:** `loss_history[0]` is the loss from the very first epoch, measured *before* the first update. How should it compare with the initial loss from Section 4.1?

In [ ]:
print("first epoch loss:", round(loss_history[0], 4))  # start and end of training
print("final epoch loss:", round(loss_history[-1], 4))

The first epoch loss should match the initial loss (give or take a tiny GPU rounding difference). Same weights, same data, and dropout is set to 0, so there's nothing to make them differ. The loss is computed *before* `optimizer.step()` changes anything.

The final loss is the interesting one. A big drop means gradients made it all the way through the head, the decoder, both embeddings, and the patch projection. But again, it only tells us about these sixteen examples.

### 4.3 Read the learning curve

Each point is the loss on all sixteen pairs at one epoch. Look for the overall trend going down. Don't worry if a few neighboring points wiggle up.

In [ ]:
fig, axis = plt.subplots(figsize=(6.5, 3.5))  # one loss value per epoch
axis.plot(range(1, EPOCHS + 1), loss_history, color="tab:blue")
axis.set(xlabel="epoch", ylabel="next-token loss", title="Captioner training loss")
axis.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 5. How does it write a caption on its own?

Before we generate, here's a picture of what cross-attention does while a model writes. *Show, Attend and Tell* ([Xu et al., 2015](https://arxiv.org/abs/1502.03044)) added attention over image regions to a captioner and drew where it looked for each word:

<img src="../assets/diagrams/notebook7/sat_fig2.png" alt="Attention over time while generating the caption 'A bird flying over a body of water'" width="800">

*Figure 2 from Xu et al. (2015). The top row is "soft" attention, a weighted average over the whole image, which is the kind our decoder uses.*

While writing "bird", it looks at the bird; while writing "water", it spreads out over the water. Their model is an LSTM and ours is a tiny Transformer with only 16 patches, but the cross-attention plays the same role.

### 5.1 One token at a time

During training, teacher forcing handed the model the correct previous token at every step. When we generate a caption for real, there's no correct caption to hand over. So the model has to feed itself.

This is called **autoregressive** generation:

1. Start with just BOS.
2. Predict the next token and pick the highest-scoring one (that's called *greedy* decoding).
3. Stick it on the end and repeat with the longer sequence.

We also block PAD so it can never be picked. It's filler, not caption text.

> **Something to chew on:** In training, the model never saw its own mistakes; teacher forcing always fed it the right tokens. Now, if it picks one bad word early, every later step builds on that bad word. What do you think that does to longer captions?

In [ ]:
@torch.inference_mode()  # each picked token gets fed back in as part of the next input
def generate(image_batch):
    captioner.eval()
    generated = torch.full((len(image_batch), 1), BOS_ID, device=device)  # start with just BOS
    for _ in range(MAX_LENGTH - 1):
        next_scores = caption_forward(image_batch, generated)[:, -1]  # scores for the next token only
        next_scores[:, PAD_ID] = -torch.inf  # PAD is never allowed in a caption
        next_id = next_scores.argmax(dim=-1, keepdim=True)  # greedy: take the top score
        generated = torch.cat((generated, next_id), dim=1)
    return generated

Notice the loop doesn't stop at EOS. It always runs `MAX_LENGTH - 1` = 17 steps, so whatever comes after EOS is junk we'll throw away next.

### 5.2 Turn IDs back into words

This helper drops BOS, cuts everything off at the first EOS, and lets the tokenizer glue the subword pieces back into a sentence. What if the model never predicts EOS? We tack one extra EOS onto the end of the list before searching, so `.index` always finds one and just keeps the whole sequence.

In [ ]:
def decode_caption(token_ids):  # the extra EOS covers sequences that never predicted one
    token_ids = [int(index) for index in token_ids]
    end_position = (token_ids + [EOS_ID]).index(EOS_ID)
    return tokenizer.decode(token_ids[1:end_position], skip_special_tokens=True).strip()

### 5.3 Compare with the reference

These four images are from our training set, so the model has seen them 300 times. Read each reference and generated caption side by side. If they're close, great: the whole path from pixels to words works. Just don't call it a good captioner. It's memorized sixteen examples, that's all.

In [ ]:
generated_ids = generate(images[:4])  # same row order as captions
for index, token_ids in enumerate(generated_ids):
    print("REFERENCE:", captions[index])
    print("GENERATED:", decode_caption(token_ids))
    print()

Let's put the captions back with their pictures. Two separate things to check: do the nouns and colors match the photo (that's the image side doing its job), and does the sentence end cleanly (that's the model learning EOS)?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.8))  # each panel: photo, reference, generated caption
for index, axis in enumerate(axes):
    axis.imshow(images[index].detach().cpu().permute(1, 2, 0))
    generated = decode_caption(generated_ids[index])
    title = "ref: " + captions[index] + chr(10) + "gen: " + generated
    axis.set_title(fill(title, 26), fontsize=8)
    axis.axis("off")
fig.suptitle("Cross-attention captioning on observed pairs")
fig.tight_layout()
plt.show()

## Recap

That's the whole trip. Here's what happened to our images and captions:

```text
16 photos, 32 × 32 RGB              [16, 3, 32, 32]
      ↓ unfold into 8 × 8 patches
16 patches of 192 values each       [16, 16, 192]
      ↓ linear layer + image positions
image memory                        [16, 16, 48]

16 captions as token IDs            [16, 18]
      ↓ shift by one (teacher forcing)
decoder input / next-token target   [16, 17] each
      ↓ token embedding + text positions
text vectors                        [16, 17, 48]

text vectors + image memory
      ↓ masked self-attention on text, cross-attention to all 16 patches
      ↓ vocabulary head
next-token scores                   [16, 17, vocabulary]
      ↓ train with cross-entropy, then generate greedily from BOS
caption text
```

**The big ideas:**

- A caption becomes token IDs with **BOS**, **EOS**, and **PAD**. A separate PAD lets us ignore filler in the loss while still teaching the model to predict EOS.
- Training shifts the caption by one: each position predicts the next token, always given the true previous tokens (**teacher forcing**).
- **Cross-attention** is the bridge: text positions ask the questions (queries), image patches provide the answers (keys and values).
- The causal mask controls which *words* each position can see, while the *whole image* stays visible at every step.
- Generation is **autoregressive**: the model feeds its own predictions back in, one token at a time.
- Fitting sixteen training pairs shows the code works end to end. It says nothing about captioning photos the model hasn't seen.

### Check your understanding

1. Why do we add a separate PAD token instead of reusing EOS for padding?
2. A caption is `[BOS, a, red, bus, EOS, PAD]`. What are the decoder input and the target after the shift?
3. The causal mask hides future text tokens. Why don't we put a similar mask on the image patches?

<details>
<summary>Show answers</summary>

1. The loss ignores PAD targets with `ignore_index=PAD_ID`. If PAD and EOS shared an ID, we'd ignore every EOS too, and the model would never learn when to stop.
2. Input: `[BOS, a, red, bus, EOS]`. Target: `[a, red, bus, EOS, PAD]`. The last target is PAD, so the loss skips it.
3. The causal mask exists so a position can't peek at the word it's supposed to predict. The image isn't something we're predicting; it's given in full from the start, both in training and in generation. So every text position is free to read all 16 patches.

</details>

## Try it yourself

None of these retrain anything. They all reuse the captioner you just trained, so each one takes a second or two.

### TODO 1: Caption a photo the model has never seen

Our model trained on the first sixteen distinct photos. The cell below grabs a photo from *after* those sixteen, preps it the same way, and generates a caption.

Before you run it, guess: will the caption describe this new photo, or will it sound like one of the sixteen training captions?

In [ ]:
# TODO: change other_position. 0 to 15 are training photos; PAIR_COUNT (16) and up are new to the model
other_position = PAIR_COUNT
all_ids = list(first_row)
other_cocoid = all_ids[min(other_position, len(all_ids) - 1)]
other_image = image_transform(dataset[first_row[other_cocoid]]["image"].convert("RGB")).unsqueeze(0).to(device)
other_generated = decode_caption(generate(other_image)[0])

fig, axis = plt.subplots(figsize=(3.5, 3.8))
axis.imshow(other_image[0].cpu().permute(1, 2, 0))
axis.set_title(fill("ref: " + first_caption[other_cocoid] + chr(10) + "gen: " + other_generated, 30), fontsize=8)
axis.axis("off")
fig.tight_layout()
plt.show()

Try a few new photos. Don't be surprised if the generated caption sounds suspiciously like one of the training captions. With only sixteen examples, the model has basically learned sixteen sentences, and it reaches for whichever one fits best. Then set `other_position` back to something from 0 to 15 and compare.

<details>
<summary>Hint</summary>

To see which training caption it's echoing, print them all with `for index, caption in enumerate(captions): print(index, caption)`.

</details>

### TODO 2: Cut the caption short

`generate` always runs `MAX_LENGTH - 1` steps. The cell below is the same greedy loop, but you pick how many steps it gets with `my_max_steps`. Try a few values and watch where captions get cut off.

In [ ]:
# TODO: try my_max_steps = 2, 5, 10, ...
my_max_steps = 5
my_steps = min(my_max_steps, MAX_LENGTH - 1)  # text_position only has MAX_LENGTH slots, so don't go past what generate() uses

with torch.inference_mode():
    my_generated = torch.full((4, 1), BOS_ID, device=device)  # same start as generate(): just BOS
    for _ in range(my_steps):
        my_scores = caption_forward(images[:4], my_generated)[:, -1]
        my_scores[:, PAD_ID] = -torch.inf
        my_generated = torch.cat((my_generated, my_scores.argmax(dim=-1, keepdim=True)), dim=1)

for index, token_ids in enumerate(my_generated):
    print(f"[{my_steps} steps] {decode_caption(token_ids)}")

Short limits chop captions mid-sentence. That's the case `decode_caption`'s extra EOS is there for. Once the limit is longer than a caption, raising it changes nothing, because everything after EOS gets thrown away anyway.

<details>
<summary>Why can't I go past 17?</summary>

`text_position` is an embedding table with `MAX_LENGTH` = 18 rows, one per position. Ask for more and the forward pass would need a position that doesn't exist, which crashes. On top of that, the model only ever *trained* on positions 0 to 16, so anything beyond that would be untrained anyway. To really allow longer captions, you'd have to raise `MAX_LENGTH` and retrain.

</details>

### TODO 3: Look at the raw tokens

`decode_caption` hides a lot: BOS, EOS, and everything after EOS. Let's look at what the model *actually* produced, token by token, and peek at its top guesses for the first word.

In [ ]:
# TODO: change my_index (0 to 3) to inspect another generated caption
my_index = 0
print("raw tokens:", tokenizer.convert_ids_to_tokens(generated_ids[my_index].tolist()))

with torch.inference_mode():
    my_first_scores = caption_forward(images[my_index:my_index + 1], generated_ids[my_index:my_index + 1, :1])[0, -1]
my_top = my_first_scores.softmax(dim=-1).topk(5)  # top 5 guesses for the token right after BOS
for prob, token_id in zip(my_top.values.tolist(), my_top.indices.tolist()):
    print(f"{tokenizer.convert_ids_to_tokens(token_id):>15}  {prob:.3f}")

Find EOS in the raw tokens. What comes after it? Remember, `generate` keeps going for the full 17 steps no matter what, so the tail can be anything.

Then look at the top-5 list. Is the model very sure about its first word, or is it torn between a few options? If training went well, it's probably pretty confident on these, since it's seen these exact captions 300 times.

<details>
<summary>Hint</summary>

Want the top guesses at a later step instead? Swap `:1` for `:k` in `generated_ids[my_index:my_index + 1, :1]` to feed the first `k` tokens, and you'll get the guesses for token number `k`. Keep `k` at most 17.

</details>

---

**Up next, Notebook 08:** instead of writing a free-form sentence, the model gets a *question* about an image and picks an answer from a small set. Welcome to visual question answering.